# Thesis — Step 1: Data Cleaning

**Thesis:** Machine Learning Analysis of Household Food Insecurity and Vulnerability Profiles in Afghanistan

**What this notebook does**

1. Loads the FAO DIEM rounds 4 to 8 for Afghanistan
2. Harmonises the outcome variable (FCG) across rounds
3. Keeps only structural predictors, so no other food security indicator enters the model
4. Handles missing values
5. Saves one clean file per round


## 1. Imports

In [17]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)
print('pandas', pd.__version__)

pandas 3.0.3


## 2. The data files

These are the five rounds that contain the FCS module. Round 3 does not have it, so it is not used here.

In [18]:
FILES = {
    4: ('data_anon.xlsx',       'excel'),   # March and April 2022, 25 provinces
    5: ('anon_dfAFG_R5.xlsx',   'excel'),   # July and August 2022, 25 provinces
    6: ('data_anon_AFG_R6.csv', 'csv'),     # January and February 2023, 26 provinces
    7: ('anon_df_AFG_R7.csv',   'csv'),     # September and October 2023, 34 provinces
    8: ('anon_dfAFG_R8.xlsx',   'excel'),   # January and February 2024, 34 provinces
}

def load(path, kind):
    if kind == 'excel':
        return pd.read_excel(path)
    return pd.read_csv(path, low_memory=False)

## 3. The outcome variable

FCG, the Food Consumption Group, has three classes: Poor, Moderate and Acceptable.

Rounds 4, 5 and 6 store it as numbers, rounds 7 and 8 store it as text. This mapping makes them identical.

In [19]:
FCG_MAP = {
    1: 'Poor', 2: 'Moderate', 3: 'Acceptable',
    1.0: 'Poor', 2.0: 'Moderate', 3.0: 'Acceptable',
    'Poor': 'Poor', 'Moderate': 'Moderate', 'Acceptable': 'Acceptable',
}

## 4. The predictor variables

Excluded on purpose, because using them would mean predicting food insecurity from food insecurity: FIES items and probabilities, HHS, HHG, rCSI, LCSI, the coping items, HDDS, and the food group frequencies that make up the FCS itself.

Kept, structural and exogenous factors only.

In [20]:
DEMO = ['hh_gender', 'hh_education', 'hh_maritalstat', 'hh_residencetype',
        'hh_wealth_water', 'hh_wealth_toilet', 'hh_wealth_light', 'hh_agricactivity']

LIVE = ['income_main', 'income_sec', 'income_main_comp', 'tot_income',
        'resp_iscropproducer', 'resp_islsproducer', 'main_agric_source',
        'crp_landsize_ha', 'crp_main', 'crp_irrig_source',
        'crp_area_change', 'crp_harv_change',
        'ls_main', 'ls_num_diff']

GEO = ['adm1_name']

print('demographics:', len(DEMO), ' livelihoods:', len(LIVE), ' geography:', len(GEO))

demographics: 8  livelihoods: 14  geography: 1


## 5. The cleaning function

A note on need_food. A household saying it needs food aid is very close to the outcome itself, so it is excluded from the main model. It can be put back later as a robustness check by passing drop_need_food=False.

In [21]:
def clean_round(path, kind, rnd, drop_need_food=True):
    df = load(path, kind)
    out = pd.DataFrame(index=df.index)

    # outcome
    out['fcg'] = df['fcg'].map(FCG_MAP)

    # shocks, without the no shock flag since it is the complement of the others
    shocks = [c for c in df.columns if c.startswith('shock_')
              and not c.endswith(('_dk', '_ref', '_other', '_otherspecify'))
              and c != 'shock_noshock']

    # assistance received
    assist = [c for c in df.columns if c.startswith('need_received_')
              and not c.endswith(('_dk', '_ref', '_other', '_otherspecify'))]

    # needs expressed
    needs = [c for c in df.columns if c.startswith('need_')
             and not c.startswith('need_received_')
             and not c.endswith(('_dk', '_ref', '_other', '_otherspecify'))
             and c != 'need']
    if drop_need_food:
        needs = [c for c in needs if c != 'need_food']

    keep = [c for c in DEMO + LIVE + GEO + shocks + assist + needs if c in df.columns]
    out = pd.concat([out, df[keep]], axis=1)

    # derived variable, how many shocks the household reported
    out['n_shocks'] = df[[c for c in shocks if c in df.columns]].eq('Yes').sum(axis=1)

    # survey weight and round number
    out['weight_final'] = df['weight_final'] if 'weight_final' in df.columns else 1.0
    out['round'] = rnd

    # drop households with no outcome value
    out = out[out['fcg'].notna()].copy()

    # numeric missing values, fill with the median and keep a flag
    num = out.select_dtypes(include=[np.number]).columns.drop(['round', 'n_shocks'], errors='ignore')
    for c in num:
        if out[c].isna().any():
            out[c + '_missing'] = out[c].isna().astype(int)
            out[c] = out[c].fillna(out[c].median())

    # categorical missing values are skip logic, for example livestock questions
    # were never asked of households without livestock, so they become a category
    cat = out.select_dtypes(include=['object', 'string']).columns.drop('fcg', errors='ignore')
    for c in cat:
        out[c] = out[c].fillna('Not applicable')

    return out

print('function defined')

function defined


## 6. Run the cleaning on all rounds

In [22]:
clean = {}
for rnd, (fn, kind) in FILES.items():
    clean[rnd] = clean_round(fn, kind, rnd)
    d = clean[rnd]
    print("Round", rnd, ":", d.shape[0], "households,", d.shape[1], "columns,", d.isna().sum().sum(), "missing cells")
    print("   ", dict(d['fcg'].value_counts()))

Round 4 : 5688 households, 74 columns, 0 missing cells
    {'Poor': np.int64(3601), 'Moderate': np.int64(1306), 'Acceptable': np.int64(781)}
Round 5 : 6019 households, 75 columns, 0 missing cells
    {'Poor': np.int64(2966), 'Moderate': np.int64(1569), 'Acceptable': np.int64(1484)}
Round 6 : 6082 households, 76 columns, 0 missing cells
    {'Poor': np.int64(3449), 'Moderate': np.int64(1532), 'Acceptable': np.int64(1101)}
Round 7 : 9560 households, 78 columns, 0 missing cells
    {'Poor': np.int64(5079), 'Moderate': np.int64(2847), 'Acceptable': np.int64(1634)}
Round 8 : 9494 households, 77 columns, 0 missing cells
    {'Moderate': np.int64(4115), 'Poor': np.int64(3388), 'Acceptable': np.int64(1991)}


## 7. Look at the cleaned Round 8

In [23]:
r8 = clean[8]
r8.head()

,fcg,hh_gender,hh_education,hh_maritalstat,hh_residencetype,hh_wealth_water,hh_wealth_toilet,hh_wealth_light,hh_agricactivity,income_main,income_sec,income_main_comp,tot_income,resp_iscropproducer,resp_islsproducer,crp_landsize_ha,crp_main,crp_irrig_source,crp_area_change,crp_harv_change,ls_main,ls_num_diff,adm1_name,shock_sicknessordeathofhh,shock_lostemplorwork,shock_otherintrahhshock,shock_higherfoodprices,shock_higherfuelprices,shock_mvtrestrict,shock_othereconomicshock,shock_pestoutbreak,shock_plantdisease,shock_animaldisease,shock_napasture,shock_othercropandlivests,shock_coldtemporhail,shock_flood,shock_hurricane,shock_drought,shock_earthquake,shock_landslides,shock_firenatural,shock_othernathazard,shock_violenceinsecconf,shock_theftofprodassets,shock_firemanmade,shock_othermanmadehazard,need_received_food,need_received_cash,need_received_vouchers_fair,need_received_crop_assist,need_received_ls_assist,need_received_fish_assist,need_received_rehabilitation,need_received_sales_support,need_received_none,need_cash,need_vouchers_fair,need_crop_inputs,need_crop_infrastructure,need_crop_knowledge,need_ls_feed,need_ls_vet_service,need_ls_infrastructure,need_ls_knowledge,need_fish_inputs,need_fish_infrastructure,need_fish_knowledge,need_env_infra_rehab,need_cold_storage,need_marketing_supp,n_shocks,weight_final,round,tot_income_missing,crp_landsize_ha_missing,ls_num_diff_missing
0,Poor,Male,"Completed higher education (university, colleg...",Married,Permanent resident,Protected well,Traditional pit latrine (no water),Solar panel,Yes - crop production,Production and sale of staple crops,No second most important source of income,Slightly less,20000.000000,Yes,No,1.200000,Wheat,"Natural surface water (river, lake, stream, po...",Same,Slightly less,Not applicable,-1.0,Kapisa,No,No,No,Yes,No,No,Yes,No,Yes,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,No,No,Yes,No,Yes,No,No,No,No,No,No,No,No,Yes,No,3,0.441876,8,0,0,1
1,Poor,Male,"Completed higher education (university, colleg...",Married,Permanent resident,"Surface water (e.g. canal, ponds, dams)",Traditional pit latrine (no water),Solar panel,Yes - crop production,Production and sale of staple crops,No second most important source of income,Slightly less,15000.000000,Yes,No,0.400000,Wheat,"Natural surface water (river, lake, stream, po...",Same,Slightly less,Not applicable,-1.0,Kapisa,No,No,No,Yes,No,No,Yes,No,No,No,No,No,No,No,No,Yes,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,Yes,No,No,No,Yes,No,No,No,No,No,No,No,No,No,No,3,0.660226,8,0,0,1
2,Moderate,Male,"Completed higher education (university, colleg...",Married,Permanent resident,Protected well,Traditional pit latrine (no water),Electricity (from generator or electrical grid),Yes - both crop and livestock production,Production and sale of staple crops,No second most important source of income,Same,10000.000000,Yes,Yes,0.566667,Wheat,"Natural surface water (river, lake, stream, po...",Slightly less,Slightly less,"Cattle (cow, beef, veal, yak, buffalo)",-2.0,Kapisa,Yes,No,No,Yes,No,No,No,No,Yes,No,No,Yes,No,No,No,Yes,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,Yes,No,No,No,Yes,No,No,No,No,No,No,No,No,No,No,5,0.441876,8,0,0,0
3,Poor,Male,Basic education,Married,Permanent resident,Protected well,Traditional pit latrine (no water),Electricity (from generator or electrical grid),Yes - crop production,Production and sale of vegetables or fruit,No second most important source of income,Slightly less,24333.333333,Yes,No,0.400000,Onions,Groundwater,Slightly less,Same,Not applicable,-1.0,Kabul,No,Yes,No,No,No,No,Yes,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,Yes,No,No,No,No,No,No,No,No,No,No,No,Yes,No,No,2,5.005188,8,0,0,1
4,Poor,Male,No education,Married,Permanent resident,Protected well,Traditional pit latrine (no water),Electricity (from generator or electrical grid),Yes - crop production,Production and sale of staple crops,No second most important source o

In [24]:
unweighted = r8['fcg'].value_counts(normalize=True) * 100

weighted = r8.groupby('fcg')['weight_final'].sum()
weighted = weighted / weighted.sum() * 100

comparison = pd.DataFrame({'unweighted %': unweighted.round(1), 'weighted %': weighted.round(1)})
print(comparison)

            unweighted %  weighted %
fcg                                 
Acceptable          21.0        19.9
Moderate            43.3        47.0
Poor                35.7        33.1


In [25]:
shock_cols = [c for c in r8.columns if c.startswith('shock_')]
top_shocks = (r8[shock_cols].eq('Yes').mean() * 100).sort_values(ascending=False)
print(top_shocks.head(10).round(1))

shock_drought                63.2
shock_higherfoodprices       58.4
shock_plantdisease           44.2
shock_sicknessordeathofhh    39.2
shock_othereconomicshock     24.3
shock_coldtemporhail         19.7
shock_lostemplorwork         19.4
shock_otherintrahhshock      17.5
shock_pestoutbreak           16.2
shock_higherfuelprices       16.1
dtype: float64


## 8. Save the clean files

In [26]:
for rnd, d in clean.items():
    d.to_csv('clean_R' + str(rnd) + '.csv', index=False)
    print('saved clean_R' + str(rnd) + '.csv')

saved clean_R4.csv
saved clean_R5.csv
saved clean_R6.csv
saved clean_R7.csv
saved clean_R8.csv


## 9. Variable list for the methodology chapter

This prints the final predictor list grouped by type, ready to describe in the thesis.

In [27]:
preds = [c for c in r8.columns if c not in ('fcg', 'weight_final', 'round')]

groups = {
    'Demographics':        [c for c in preds if c.startswith('hh_')],
    'Livelihoods':         [c for c in preds if c.startswith(('income_', 'crp_', 'ls_', 'resp_', 'main_agric', 'tot_'))],
    'Shocks':              [c for c in preds if c.startswith('shock_') or c == 'n_shocks'],
    'Needs':               [c for c in preds if c.startswith('need_') and not c.startswith('need_received_')],
    'Assistance received': [c for c in preds if c.startswith('need_received_')],
    'Geography':           [c for c in preds if c.startswith('adm')],
}

for name, cols in groups.items():
    print(name, '(' + str(len(cols)) + '):')
    for c in cols:
        print('   ', c)
    print()

print('TOTAL PREDICTORS:', len(preds))

Demographics (8):
    hh_gender
    hh_education
    hh_maritalstat
    hh_residencetype
    hh_wealth_water
    hh_wealth_toilet
    hh_wealth_light
    hh_agricactivity

Livelihoods (16):
    income_main
    income_sec
    income_main_comp
    tot_income
    resp_iscropproducer
    resp_islsproducer
    crp_landsize_ha
    crp_main
    crp_irrig_source
    crp_area_change
    crp_harv_change
    ls_main
    ls_num_diff
    tot_income_missing
    crp_landsize_ha_missing
    ls_num_diff_missing

Shocks (25):
    shock_sicknessordeathofhh
    shock_lostemplorwork
    shock_otherintrahhshock
    shock_higherfoodprices
    shock_higherfuelprices
    shock_mvtrestrict
    shock_othereconomicshock
    shock_pestoutbreak
    shock_plantdisease
    shock_animaldisease
    shock_napasture
    shock_othercropandlivests
    shock_coldtemporhail
    shock_flood
    shock_hurricane
    shock_drought
    shock_earthquake
    shock_landslides
    shock_firenatural
    shock_othernathazard
    shock_

## 10. Weighted shares of the most common shocks

The shares in section 7 describe the households interviewed. With the survey weights they describe the rural population, so the weighted shares are the ones reported in the thesis.

In [28]:
w = r8['weight_final']
shock_table = pd.DataFrame({
    'unweighted %': r8[shock_cols].eq('Yes').mean() * 100,
    'weighted %': r8[shock_cols].eq('Yes').mul(w, axis=0).sum() / w.sum() * 100,
}).round(1)
print(shock_table.sort_values('weighted %', ascending=False).head(10))

                           unweighted %  weighted %
shock_higherfoodprices             58.4        60.4
shock_drought                      63.2        58.4
shock_sicknessordeathofhh          39.2        41.8
shock_plantdisease                 44.2        36.7
shock_othereconomicshock           24.3        24.2
shock_lostemplorwork               19.4        22.9
shock_coldtemporhail               19.7        20.2
shock_higherfuelprices             16.1        19.7
shock_pestoutbreak                 16.2        17.9
shock_otherintrahhshock            17.5        17.5


**What the result shows.** With the survey weights, higher food prices become the most common shock in Round 8 (60.4 percent), just ahead of drought (58.4 percent). Without weights the order is reversed, with drought at 63.2 percent. Drought and plant disease are more common among the households interviewed than in the rural population they represent, so the sample is somewhat more exposed to agricultural shocks than the population. The same shocks are at the top in both versions, and the thesis reports the weighted shares.